# ArcRho API Smoke Test

Use this notebook to test the local `arcrho_api` package against an existing ArcRho Server project and DFM method.

Read checks are safe by default. Write checks are disabled until `ENABLE_WRITE_TESTS = True`.

In [ ]:
from pathlib import Path
import json
import sys

# Make the notebook import the package from this repo without installing it.
repo_pkg = Path.cwd()
if repo_pkg.name.lower() == "examples":
    repo_pkg = repo_pkg.parent
src_path = repo_pkg / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from arcrho_api import ArcRhoClient
from arcrho_api.migration import ArcRhoSession

print("Using package source:", src_path)

## Configure Test Target

Edit these values before running the rest of the notebook.

In [ ]:
SERVER_ROOT = r"E:\\ArcRho Server"
PROJECT_NAME = ""          # Example: "2026Q1 Reserve Review"
RESERVING_CLASS = ""       # Example: r"Auto\\Private Passenger"
DFM_NAME = ""              # Example: "Paid Loss Ultimate"

# Keep this False until you intentionally want to write JSON files.
ENABLE_WRITE_TESTS = False

print("SERVER_ROOT:", SERVER_ROOT)
print("PROJECT_NAME:", PROJECT_NAME or "<choose from project list>")
print("RESERVING_CLASS:", RESERVING_CLASS or "<choose from DFM list>")
print("DFM_NAME:", DFM_NAME or "<choose from DFM list>")

## List Projects

In [ ]:
client = ArcRhoClient(SERVER_ROOT, read_only=True)
projects = client.list_projects()
print(f"Found {len(projects)} projects")
projects[:25]

## Open Project And List DFM Methods

In [ ]:
assert PROJECT_NAME, "Set PROJECT_NAME first. Pick one from the project list above."

project = client.project(PROJECT_NAME)
settings = project.settings()
print("Project path:", settings.project_path)
print("General settings keys:", sorted(settings.general_settings.keys())[:20])

dfm_refs = project.list_dfm_methods(refresh=False)
print(f"Found {len(dfm_refs)} DFM methods")
[(item.path, item.name) for item in dfm_refs[:50]]

## Open Existing DFM

In [ ]:
assert RESERVING_CLASS and DFM_NAME, "Set RESERVING_CLASS and DFM_NAME from the list above."

rc = project.reserving_class(RESERVING_CLASS)
dfm = rc.dfm(DFM_NAME)

dfm.info()

## Inspect DFM Details And JSON Sections

In [ ]:
print("Output vector:", dfm.output_vector)
print("Input triangle:", dfm.input_triangle)
print("Origin length:", dfm.origin_length)
print("Development length:", dfm.development_length)
print("Decimal places:", dfm.decimal_places)
print("Last modified:", dfm.last_modified)
print("Notes preview:", dfm.notes[:500])

payload = dfm.to_dict()
print("Top-level keys:", list(payload.keys()))
print("Details tab:")
payload.get("details tab", {})

## Test In-Memory DFM Helper Methods

This cell mutates only the in-memory `dfm` object. It does not write unless you call `save()`.

In [ ]:
work = rc.dfm(DFM_NAME)  # reload a fresh copy
before_notes = work.notes

# These exercise production-style helpers. Some depend on ratio data existing in the DFM JSON.
helper_results = {}
for label, fn in [
    ("clear", lambda: work.clear()),
    ("add_notes", lambda: work.add_notes("Smoke test note - not saved.")),
    ("set_selected_average", lambda: work.set_selected_average("Simple - 3")),
    ("exclude_covid_years", lambda: work.exclude_covid_years()),
    ("exclude_high", lambda: work.exclude_high(1, 1, "Smoke test high exclusion.")),
    ("exclude_low", lambda: work.exclude_low(1, 1, "Smoke test low exclusion.")),
]:
    try:
        fn()
        helper_results[label] = "ok"
    except Exception as exc:
        helper_results[label] = f"{type(exc).__name__}: {exc}"

print("Original notes unchanged on disk until save.")
helper_results

## Test Legacy Migration Session Style

In [ ]:
session = ArcRhoSession(SERVER_ROOT, read_only=True)
session.set_project(PROJECT_NAME)
session.set_reserving_class(RESERVING_CLASS)

legacy_dfm = session.DFM(DFM_NAME)
print(legacy_dfm.info())

# Legacy aliases are available for notebook migration.
print("Has ex_COVID_AY:", hasattr(legacy_dfm, "ex_COVID_AY"))
print("Has set_selected_estimate:", hasattr(legacy_dfm, "set_selected_estimate"))

## Optional Write Test

This creates or updates a small DFM JSON under the selected project/reserving class. It runs only when `ENABLE_WRITE_TESTS = True`.

In [ ]:
if not ENABLE_WRITE_TESTS:
    print("Write tests skipped. Set ENABLE_WRITE_TESTS = True to run this cell.")
else:
    write_client = ArcRhoClient(SERVER_ROOT, read_only=False)
    write_project = write_client.project(PROJECT_NAME)
    write_rc = write_project.reserving_class(RESERVING_CLASS)
    smoke_name = "API Smoke Test DFM"
    smoke = write_rc.new_dfm(
        smoke_name,
        output_vector=smoke_name,
        input_triangle=dfm.input_triangle or "Input Triangle",
        origin_length=dfm.origin_length or 12,
        development_length=dfm.development_length or 12,
        notes="Created by arcrho_api smoke test.",
    )
    path = smoke.save()
    print("Saved:", path)
    print("Index count:", len(write_project.list_dfm_methods(refresh=True)))